# Emotion Detection

Given an image of a facial expression, we want to determine what emotion is being portrayed. This is a multiclass classification problem as there are a total of 7 different emotions that we can classify instances as: angry, disgust, fear, happy, sad, surprised, and neutral. Our goal is to build and compare different Convolutional Neural Network (CNN) models to classify these emotions from inputted images.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import pickle

## Load Preprocessed Data

In [ ]:
data = np.load('../data/processed_data.npz')

X_train = data['X_train']
X_test = data['X_test']
X_val = data['X_val']

X_train_normalized = data['X_train_normalized']
X_test_normalized = data['X_test_normalized']
X_val_normalized = data['X_val_normalized']

y_train = data['y_train']
y_test = data['y_test']
y_val = data['y_val']

y_train_cat = data['y_train_cat']
y_test_cat = data['y_test_cat']
y_val_cat = data['y_val_cat']

## Build Models


Build a Convolutional Neural Network (CNN) that classifies images by 1 of 7 emotion types.

### Model 1 (Baseline)

Start by creating a simple model with only 1 convolutional layer.

In [ ]:
model1 = tf.keras.Sequential([
    # define input shape, 48x48 pixels, 1 channel (grayscale)
    layers.Input(shape=(48, 48, 1)),

    # convolutional layer
    layers.Conv2D(32, (3, 3), activation='relu'),

    # convert 2D feature map into a 1D vector
    layers.Flatten(),

    # fully connected layer that learns combinations of the features detected by the convolutional layer, 64 neurons = 64 combinations/patterns
    layers.Dense(64, activation='relu'),

    # final layer with 7 neurons (one for each emotion)
    layers.Dense(7, activation='softmax')
])

model1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history1 = model1.fit(
    X_train_normalized, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_normalized, y_val_cat),
    callbacks=[early_stop]
)

# save history during training
with open('../models/history1.pkl', 'wb') as f:
    pickle.dump(history1.history, f)

test_accuracy1, test_loss1 = model1.evaluate(X_test_normalized, y_test_cat, verbose=0)

model1.save('../models/model_v1.keras')

In [ ]:
## plot results

plt.figure(figsize=(12, 5))

# accuracy
plt.subplot(1, 2, 1)
plt.plot(history1.history['accuracy'], label='train')
plt.plot(history1.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# loss
plt.subplot(1, 2, 2)
plt.plot(history1.history['loss'], label='train')
plt.plot(history1.history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

# test results
test_accuracy, test_loss = model.evaluate(X_test_normalized, y_test_cat, verbose=0)
print(f'Test Accuracy: {test_accuracy:.4f} | Test Loss: {test_loss:.4f}')

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# confusion matrix
y_pred = np.argmax(model.predict(X_test_normalized), axis=1)
y_true = np.argmax(y_test_cat, axis=1)
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=emotion_labels).plot(cmap='Blues', xticks_rotation='vertical')
plt.title('Confusion Matrix')
plt.show()

# classification report
print(classification_report(y_true, y_pred, target_names=emotion_labels))

Model 1 is overfitting, it does well on training data but poorly on validation data. Model is too simple/shallow, only 1 convolutional layer and 1 dense layer is not enough to learn and capture complexity of facial features.

The happy emotion was the easiest class to detect whereas disgust was completely missed. Since certain emotions have more images than others, we can possibly introduce class weights or perform data augmentation to address the class imbalance.

Overall accuracy = 41%, F1 score = 0.40